# Segmentation d'images avec PyTorch Lightning

Détourer les forêts sur des photographies aériennes de l'IGN : c'est un problème de segmentation, où le modèle classe chaque pixel, forêt ou non-forêt. Ce notebook entraîne et compare quatre modèles, de trois architectures présentées dans les slides « Segmentation et UNet » :

1. un UNet simple, entraîné depuis zéro ;
2. un encodeur EfficientNetV2-B0 pré-entraîné sur ImageNet, suivi d'un décodeur sans connexions de saut : ce qu'on perd sans elles ;
3. un UNet dont l'encodeur est ce même EfficientNetV2-B0 : deux modèles, l'un à l'encodeur gelé, l'autre fine-tuné.

Il demande un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Le téléchargement des données (environ 0,8 Go) prend quelques minutes : lancez les quatre cellules suivantes dès le début.

In [ ]:
!pip install -q lightning torchmetrics timm segmentation-models-pytorch

In [ ]:
import random

import lightning
import matplotlib.pyplot as plt
import numpy
import pandas
import segmentation_models_pytorch as smp
import timm
import torch
import torchmetrics
from imageio.v3 import imread
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms.v2 import functional as transforms

In [ ]:
!nvidia-smi

In [ ]:
%%time
!rm -rf sample_data
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh ign-forest-2009

## Chargement des données

Les données sont des photographies aériennes prises par l'IGN en 2009 en Lorraine : 20 zones de 5 km × 5 km. Chaque zone est découpée en 9 × 9 = 81 tuiles de 1024 × 1024 pixels (1 pixel = 50 cm sur le terrain : une tuile couvre 512 m × 512 m).

- **Entrée** (fichiers `*_irc.jpg`) : une image en *infrarouge couleur* (IRC). Ses trois canaux sont le proche infrarouge, le rouge et le vert, affichés en rouge, vert et bleu. La végétation réfléchit fortement l'infrarouge : elle apparaît en rouge.
- **Cible** (fichiers `*_pfnf_nch.png`) : un masque en noir et blanc, la classe de chaque pixel : 1 pour forêt, 0 pour non-forêt. Au sens de l'IGN, une forêt n'est pas un simple groupe d'arbres : c'est une surface de plus de 5 000 m², large d'au moins 20 m, dont la végétation couvre au moins 10 % du sol et qui n'est pas utilisée pour l'agriculture.

2 zones ont été mises de côté pour la validation, et 2 autres pour le test. Toutes les tuiles d'une zone vont dans le même ensemble : des tuiles voisines se ressemblent trop pour mesurer honnêtement la généralisation. Il reste 16 zones pour l'entraînement, soit 1 296 tuiles.

In [ ]:
df_entrainement = pandas.read_csv("ign-forest-2009/train-files.csv")
df_val = pandas.read_csv("ign-forest-2009/val-files.csv")
df_test = pandas.read_csv("ign-forest-2009/test-files.csv")

print("Coordonnées en entraînement")
print(df_entrainement.coord.unique())

print("Coordonnées en val")
print(df_val.coord.unique())

print("Coordonnées en test")
print(df_test.coord.unique())

df_test

## Étude d'un exemple et de son annotation

In [ ]:
def lit_observation(chemin: str) -> torch.Tensor:
  """Lit une tuile, en tenseur de forme (canaux, hauteur, largeur)."""
  return torch.from_numpy(imread(chemin)).permute(2, 0, 1)


def lit_annotation(chemin: str) -> torch.Tensor:
  """Lit une annotation, en masque de forme (1, hauteur, largeur)."""
  return torch.from_numpy(imread(chemin))[None].float()


exemple_entree = lit_observation(df_entrainement.Observations[0])
exemple_annotation = lit_annotation(df_entrainement.Annotations[0])

In [ ]:
def rend_affichable(image: torch.Tensor) -> numpy.ndarray:
  """Convertit un tenseur (canaux, hauteur, largeur) en tableau pour imshow."""
  image = image.detach().cpu()
  if image.shape[0] == 3:
    # Image en infrarouge couleur, canaux en dernier pour imshow
    return image.permute(1, 2, 0).numpy()
  # Cible ou prédiction : la carte de la classe forêt
  return image[-1].numpy()


def affiche_echantillon(*images: torch.Tensor) -> None:
  """Affiche côte à côte une image d'entrée, sa cible et la prédiction."""
  figure, axes = plt.subplots(1, len(images), figsize=(18, 18))

  titres = ["Entrée", "Cible", "Prédiction"]
  for image, ax, titre in zip(images, axes, titres):
    image = rend_affichable(image)
    ax.set_title(titre)
    if image.ndim == 3:
      ax.imshow(image)
    else:
      # Bornes fixes : forêt (1) en jaune, non-forêt (0) en violet, même quand
      # toute la tuile est d'une seule classe
      ax.imshow(image, vmin=0, vmax=1)
    ax.axis("off")
  plt.show()


affiche_echantillon(exemple_entree, exemple_annotation)

## Options générales

In [ ]:
# Dimension des images originales
dim_originale = 1024

# Dimension des images en entrée du réseau
dim_entree = 512

# Nombre de canaux des images en entrée
nb_canaux = 3

# Nombre de processus utilisés pour charger les données
nb_lecteurs = 2

# Nombre d'epochs du UNet entraîné depuis zéro, des modèles à encodeur
# pré-entraîné (gelé, seul leur décodeur apprend : ils convergent plus vite), puis
# du fine-tuning
nb_epoques = 15
nb_epoques_pretraine = 5
nb_epoques_affinage = 3

## Augmentation des données

Avec 1 296 tuiles d'entraînement, le réseau reverrait vite les mêmes images : on les augmente par des transformations aléatoires, qui le rendent moins sensible aux rotations, au cadrage, etc. Implémentez la fonction `transforme_donnees`, qui reçoit une image `X`, de forme (3, 1024, 1024), et sa cible `Y`, de forme (1, 1024, 1024) :

- si `evaluation` vaut `True` (validation et test), un recadrage centré de `X` et de `Y` à la taille `taille_recadree` ;
- sinon (entraînement), un recadrage aléatoire à cette taille, puis une rotation d'un nombre aléatoire de quarts de tour et des miroirs aléatoires.

La cible doit subir exactement la même transformation que l'image, sinon le masque ne correspond plus aux pixels. Vérifiez-le sur l'exemple, en décommentant la dernière ligne de la cellule. Le module `torchvision.transforms.v2.functional` est importé sous le nom `transforms`.

**Documentation utile** :

- [`transforms.crop`](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.v2.functional.crop.html)
- [`torch.rot90`](https://docs.pytorch.org/docs/stable/generated/torch.rot90.html)
- [`transforms.horizontal_flip`](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.v2.functional.horizontal_flip.html)
- [`transforms.vertical_flip`](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.v2.functional.vertical_flip.html)
- [`random.randint`](https://docs.python.org/3/library/random.html#random.randint)

In [ ]:
def transforme_donnees(
  X: torch.Tensor,
  Y: torch.Tensor,
  evaluation: bool = False,
  taille_recadree: tuple[int, int] = (dim_entree, dim_entree),
) -> tuple[torch.Tensor, torch.Tensor]:
  # Votre code ici
  return X, Y


# affiche_echantillon(*transforme_donnees(exemple_entree, exemple_annotation))

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `hauteur, largeur = X.shape[-2:]` : les dimensions de l'image sont les deux dernières du tenseur ; les tuiles font 1024 × 1024 pixels, et `taille_recadree` vaut (512, 512)
- `transforms.crop(X, haut, gauche, hauteur_recadree, largeur_recadree)` : le coin en haut à gauche du recadrage, puis sa taille
- Recadrage centré : `haut = (hauteur - hauteur_recadree) // 2` ; aléatoire : `random.randint(0, hauteur - hauteur_recadree)`, bornes comprises
- `torch.rot90(X, k, dims=(-2, -1))` : `k` quarts de tour dans le plan de l'image, avec `k = random.randint(0, 3)`
- `transforms.horizontal_flip(X)` et `transforms.vertical_flip(X)`, chacun une fois sur deux : `if random.random() < 0.5:`
- Tirez chaque paramètre aléatoire une seule fois, puis appliquez-le à `X` et à `Y`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def transforme_donnees(
  X: torch.Tensor,
  Y: torch.Tensor,
  evaluation: bool = False,
  taille_recadree: tuple[int, int] = (dim_entree, dim_entree),
) -> tuple[torch.Tensor, torch.Tensor]:
  hauteur, largeur = X.shape[-2:]
  hauteur_recadree, largeur_recadree = taille_recadree
  if evaluation:
    haut, gauche = ..., ...  # recadrage centré
  else:
    haut = random.randint(0, ...)
    gauche = random.randint(0, ...)
  X = transforms.crop(X, haut, gauche, hauteur_recadree, largeur_recadree)
  Y = transforms.crop(Y, haut, gauche, hauteur_recadree, largeur_recadree)
  if not evaluation:
    k = random.randint(0, 3)  # nombre de quarts de tour
    X, Y = torch.rot90(X, k, dims=(-2, -1)), torch.rot90(Y, k, dims=(-2, -1))
    if random.random() < 0.5:
      X, Y = ..., ...  # miroir gauche-droite
    if random.random() < 0.5:
      X, Y = ..., ...  # miroir haut-bas
  return X, Y
```

### Solution

In [ ]:
def transforme_donnees(
  X: torch.Tensor,
  Y: torch.Tensor,
  evaluation: bool = False,
  taille_recadree: tuple[int, int] = (dim_entree, dim_entree),
) -> tuple[torch.Tensor, torch.Tensor]:
  hauteur, largeur = X.shape[-2:]
  hauteur_recadree, largeur_recadree = taille_recadree
  if evaluation:
    # Recadrage centré
    haut = (hauteur - hauteur_recadree) // 2
    gauche = (largeur - largeur_recadree) // 2
  else:
    # Recadrage aléatoire
    haut = random.randint(0, hauteur - hauteur_recadree)
    gauche = random.randint(0, largeur - largeur_recadree)
  X = transforms.crop(X, haut, gauche, hauteur_recadree, largeur_recadree)
  Y = transforms.crop(Y, haut, gauche, hauteur_recadree, largeur_recadree)
  if not evaluation:
    # Rotation d'un nombre aléatoire de quarts de tour
    k = random.randint(0, 3)
    X, Y = torch.rot90(X, k, dims=(-2, -1)), torch.rot90(Y, k, dims=(-2, -1))
    # Miroirs aléatoires, la même transformation pour l'image et la cible
    if random.random() < 0.5:
      X, Y = transforms.horizontal_flip(X), transforms.horizontal_flip(Y)
    if random.random() < 0.5:
      X, Y = transforms.vertical_flip(X), transforms.vertical_flip(Y)
  return X, Y


for _ in range(3):
  affiche_echantillon(*transforme_donnees(exemple_entree, exemple_annotation))

## Création d'un `Dataset` PyTorch

PyTorch lit les données exemple par exemple, dans un `torch.utils.data.Dataset`. Écrivez la classe `DatasetSegmentation`, qui en hérite :

- `__init__` garde les chemins des images et des cibles, le mode (`evaluation`) et la taille du recadrage ;
- `__len__` renvoie le nombre de tuiles ;
- `__getitem__(index)` lit la tuile numéro `index` et sa cible avec `lit_observation` et `lit_annotation`, les transforme avec `transforme_donnees`, et renvoie deux tenseurs : l'image, de forme (3, 512, 512), aux pixels ramenés entre 0 et 1, et la cible, de forme (1, 512, 512), qui vaut 1 pour la forêt.

Un `DataLoader`, fourni plus bas, regroupe ensuite ces exemples en batchs, mélange les tuiles d'entraînement à chaque epoch et parallélise leur lecture.

**Documentation utile** :

- [`torch.utils.data.Dataset`](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.Dataset)

In [ ]:
class DatasetSegmentation(Dataset):
  """Lit les tuiles et leurs cibles, avec les transformations voulues."""

  def __init__(
    self,
    df: pandas.DataFrame,
    evaluation: bool,
    taille_recadree: tuple[int, int] = (dim_entree, dim_entree),
  ) -> None:
    """Initialise le jeu de données."""
    # Votre code ici

  def __len__(self) -> int:
    """Calcule le nombre de tuiles."""
    # Votre code ici

  def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Renvoie la tuile numéro `index` et sa cible, transformées."""
    # Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `df.Observations.to_numpy()` et `df.Annotations.to_numpy()` : les chemins des images et des cibles, numérotés à partir de 0, même quand `df` est un extrait d'un autre tableau
- `lit_observation(chemin)` : l'image, un tenseur `uint8` de forme (3, 1024, 1024) ; `lit_annotation(chemin)` : la cible, de forme (1, 1024, 1024)
- `transforme_donnees(X, Y, self.evaluation, self.taille_recadree)`
- `X.float() / 255` : les pixels entre 0 et 1, en `float32`
- `len(DatasetSegmentation(df_entrainement, evaluation=True))` doit valoir 1296, et `DatasetSegmentation(df_entrainement, evaluation=True)[0][0].shape`, `torch.Size([3, 512, 512])`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class DatasetSegmentation(Dataset):
  def __init__(
    self,
    df: pandas.DataFrame,
    evaluation: bool,
    taille_recadree: tuple[int, int] = (dim_entree, dim_entree),
  ) -> None:
    self.chemins_X = df.Observations.to_numpy()
    self.chemins_Y = ...
    self.evaluation = evaluation
    self.taille_recadree = taille_recadree

  def __len__(self) -> int:
    return ...

  def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
    X, Y = transforme_donnees(
      lit_observation(self.chemins_X[index]),
      ...,  # cible
      self.evaluation,
      self.taille_recadree,
    )
    return ..., Y  # pixels entre 0 et 1
```

### Solution

In [ ]:
class DatasetSegmentation(Dataset):
  """Lit les tuiles et leurs cibles, avec les transformations voulues."""

  def __init__(
    self,
    df: pandas.DataFrame,
    evaluation: bool,
    taille_recadree: tuple[int, int] = (dim_entree, dim_entree),
  ) -> None:
    """Initialise le jeu de données."""
    self.chemins_X = df.Observations.to_numpy()
    self.chemins_Y = df.Annotations.to_numpy()
    assert self.chemins_X.shape[0] == self.chemins_Y.shape[0]
    self.evaluation = evaluation
    self.taille_recadree = taille_recadree

  def __len__(self) -> int:
    """Calcule le nombre de tuiles."""
    return self.chemins_X.shape[0]

  def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Renvoie la tuile numéro `index` et sa cible, transformées."""
    X, Y = transforme_donnees(
      lit_observation(self.chemins_X[index]),
      lit_annotation(self.chemins_Y[index]),
      self.evaluation,
      self.taille_recadree,
    )
    # Pixels ramenés entre 0 et 1
    return X.float() / 255, Y


dataset = DatasetSegmentation(df_entrainement, evaluation=True)
X, Y = dataset[0]
len(dataset), X.shape, Y.shape

## Création du modèle

Le modèle passe d'un batch d'images de forme (batch, 3, 512, 512) à une carte de même taille, de forme (batch, 1, 512, 512) : pour chaque pixel, un *logit*, qu'une sigmoïde transforme en probabilité d'être de la forêt. Écrivez `construit_modele`, qui construit un UNet comme celui des slides « Implémenter un UNet simple en PyTorch » et « Passe avant du UNet », mais plus profond :

- `nb_niveaux` niveaux, avec `profondeur_initiale` canaux au premier, puis deux fois plus à chaque niveau, sans dépasser 512 : 32, 64, 128, 256 et 512 par défaut ;
- à chaque niveau, un `bloc_convolution` plus riche que celui des slides : deux convolutions 3 × 3, chacune suivie d'une normalisation par batch et d'une ReLU, puis un *dropout* de 10 % ;
- l'encodeur divise la résolution par deux d'un niveau au suivant, de 512 à 32 pixels de côté ; le décodeur la double à chaque niveau par une convolution transposée, comme dans les slides, et concatène la sortie de l'encodeur de même résolution (connexion de saut) ;
- en sortie, une convolution 1 × 1 à un seul canal, sans activation : la perte applique la sigmoïde pendant l'entraînement, et la prédiction l'applique explicitement.

Cette architecture vient de [l'article qui présente U-Net](https://arxiv.org/abs/1505.04597).

**Documentation utile** :

- [`nn.BatchNorm2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html)
- [`nn.ConvTranspose2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html)
- [`nn.ModuleList`](https://docs.pytorch.org/docs/stable/generated/torch.nn.ModuleList.html)

In [ ]:
def construit_modele(profondeur_initiale: int = 32, nb_niveaux: int = 5) -> nn.Module:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `profondeurs = [min(512, profondeur_initiale * 2**i) for i in range(nb_niveaux)]` : le nombre de canaux de chaque niveau
- `nn.Conv2d(canaux_entree, canaux_sortie, 3, padding="same")` : `padding="same"` garde la taille de l'image
- `nn.BatchNorm2d(canaux_sortie)` et `nn.ReLU()` après chaque convolution, puis un seul `nn.Dropout(0.1)` en fin de bloc, enchaînés par `nn.Sequential`
- `nn.MaxPool2d(2)` divise la résolution par deux ; `nn.ConvTranspose2d(dessous, profondeur, 2, stride=2)` la double, en passant de `dessous` à `profondeur` canaux
- `nn.ModuleList([...])` range les blocs de l'encodeur, les convolutions transposées et les blocs du décodeur : dans une simple liste Python, leurs poids seraient invisibles pour l'optimiseur
- `torch.cat([x, saut], dim=1)` : la connexion de saut, sur l'axe des canaux ; le bloc du décodeur reçoit donc `2 * profondeur` canaux
- `nn.Conv2d(profondeurs[0], 1, 1)` : la couche de sortie
- `construit_modele()(torch.zeros(1, 3, 64, 64)).shape` doit valoir `torch.Size([1, 1, 64, 64])`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def bloc_convolution(canaux_entree: int, canaux_sortie: int) -> nn.Sequential:
  modules = []
  for canaux in [canaux_entree, canaux_sortie]:
    modules += [
      nn.Conv2d(canaux, canaux_sortie, 3, padding="same"),
      ...,  # normalisation par batch
      ...,  # ReLU
    ]
  return nn.Sequential(*modules, ...)  # dropout


class UNet(nn.Module):
  def __init__(self, profondeur_initiale: int = 32, nb_niveaux: int = 5) -> None:
    super().__init__()
    profondeurs = [min(512, profondeur_initiale * 2**i) for i in range(nb_niveaux)]
    # Encodeur : de nb_canaux à profondeurs[0] canaux, puis d'un niveau au suivant
    self.encodeur = nn.ModuleList(
      [bloc_convolution(i, o) for i, o in zip([nb_canaux, *profondeurs], profondeurs)]
    )
    # Décodeur : du fond du « U » vers le haut, chaque niveau double la résolution
    # du niveau inférieur (de dessous à profondeur canaux), puis reçoit la connexion de
    # saut (profondeur canaux)
    niveaux = list(zip(profondeurs[:0:-1], profondeurs[-2::-1]))
    self.sur_echantillonneurs = nn.ModuleList(
      [nn.ConvTranspose2d(dessous, profondeur, ...) for dessous, profondeur in niveaux]
    )
    self.decodeur = nn.ModuleList(
      [bloc_convolution(..., profondeur) for _, profondeur in niveaux]
    )
    self.sous_echantillonneur = nn.MaxPool2d(2)
    self.tete = ...

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    sauts = []
    for i, bloc in enumerate(self.encodeur):
      x = bloc(self.sous_echantillonneur(x) if i else x)
      sauts.append(x)
    for sur_echantillonneur, bloc, saut in zip(
      self.sur_echantillonneurs, self.decodeur, reversed(sauts[:-1])
    ):
      x = bloc(torch.cat([..., saut], dim=1))  # résolution doublée
    return ...


def construit_modele(profondeur_initiale: int = 32, nb_niveaux: int = 5) -> nn.Module:
  return UNet(profondeur_initiale, nb_niveaux)
```

### Solution

In [ ]:
# Le modèle U-Net de l'article https://arxiv.org/abs/1505.04597 (voir aussi
# les slides sur le UNet)


def bloc_convolution(canaux_entree: int, canaux_sortie: int) -> nn.Sequential:
  """Deux convolutions 3 × 3 avec normalisation par batch et ReLU, puis un dropout."""
  modules = []
  for canaux in [canaux_entree, canaux_sortie]:
    modules += [
      nn.Conv2d(canaux, canaux_sortie, 3, padding="same"),
      nn.BatchNorm2d(canaux_sortie),
      nn.ReLU(),
    ]
  return nn.Sequential(*modules, nn.Dropout(0.1))


class UNet(nn.Module):
  """U-Net : un encodeur, un décodeur, et des connexions de saut entre les deux."""

  def __init__(self, profondeur_initiale: int = 32, nb_niveaux: int = 5) -> None:
    super().__init__()
    profondeurs = [min(512, profondeur_initiale * 2**i) for i in range(nb_niveaux)]
    # Encodeur : un bloc par niveau, de nb_canaux à profondeurs[0] canaux, puis d'un
    # niveau au suivant
    self.encodeur = nn.ModuleList(
      [bloc_convolution(i, o) for i, o in zip([nb_canaux, *profondeurs], profondeurs)]
    )
    # Décodeur : du fond du « U » vers le haut, une convolution transposée double
    # la résolution du niveau inférieur et passe de dessous à profondeur canaux ; le bloc
    # reçoit ensuite sa sortie concaténée à celle de l'encodeur de même résolution
    # (profondeur canaux chacune)
    niveaux = list(zip(profondeurs[:0:-1], profondeurs[-2::-1]))
    self.sur_echantillonneurs = nn.ModuleList(
      [
        nn.ConvTranspose2d(dessous, profondeur, 2, stride=2)
        for dessous, profondeur in niveaux
      ]
    )
    self.decodeur = nn.ModuleList(
      [bloc_convolution(2 * profondeur, profondeur) for _, profondeur in niveaux]
    )
    self.sous_echantillonneur = nn.MaxPool2d(2)
    # Un logit par pixel : la sigmoïde est appliquée par la perte
    self.tete = nn.Conv2d(profondeurs[0], 1, 1)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    # Encodeur : la résolution est divisée par deux d'un niveau au suivant, et la
    # sortie de chaque niveau est gardée pour la connexion de saut
    sauts = []
    for i, bloc in enumerate(self.encodeur):
      x = bloc(self.sous_echantillonneur(x) if i else x)
      sauts.append(x)
    # Décodeur : la résolution est doublée, puis la sortie de l'encodeur de même
    # résolution est concaténée (le fond du « U », sauts[-1], n'en a pas)
    for sur_echantillonneur, bloc, saut in zip(
      self.sur_echantillonneurs, self.decodeur, reversed(sauts[:-1])
    ):
      x = bloc(torch.cat([sur_echantillonneur(x), saut], dim=1))
    return self.tete(x)


def construit_modele(profondeur_initiale: int = 32, nb_niveaux: int = 5) -> nn.Module:
  return UNet(profondeur_initiale, nb_niveaux)


construit_modele()(torch.zeros(1, nb_canaux, 64, 64)).shape

## Entraînement

La cellule suivante fournit le `LightningModule` du notebook, `SegmenteurImages` : il décrit la perte, les métriques et l'optimiseur de n'importe quel modèle de segmentation binaire. Le modèle renvoie des logits : la perte `binary_cross_entropy_with_logits` leur applique la sigmoïde pendant l'entraînement, et `predict_step` l'applique explicitement.

On suit l'exactitude (*accuracy*), la part des pixels bien classés, et l'IoU de la classe forêt (slide « Pertes et métriques en segmentation ») : la surface où la prédiction et la cible disent toutes deux « forêt », divisée par la surface où au moins l'une des deux le dit. Elle vaut 1 pour un masque parfait, et 0 pour un modèle qui ne prédit jamais la forêt, quelle que soit son exactitude : c'est sur elle qu'on choisit le meilleur modèle.

In [ ]:
class SegmenteurImages(lightning.LightningModule):
  """Module Lightning des modèles de segmentation binaire."""

  def __init__(self, modele: nn.Module, taux_apprentissage: float = 1e-4) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["modele"])
    self.modele = modele
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, nb_canaux, dim_entree, dim_entree)
    # torchmetrics demande une instance de métrique par étape
    self.exactitudes = nn.ModuleDict(
      {
        f"exactitude_{etape}": torchmetrics.Accuracy(task="binary")
        for etape in ("entrainement", "val", "test")
      }
    )
    # IoU (indice de Jaccard) de la classe positive : contrairement à l'exactitude,
    # elle n'est pas gonflée par les pixels d'une classe majoritaire
    self.ious = nn.ModuleDict(
      {
        f"iou_{etape}": torchmetrics.classification.BinaryJaccardIndex()
        for etape in ("entrainement", "val", "test")
      }
    )

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.modele(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], etape: str) -> torch.Tensor:
    images, annotations = batch
    logits = self(images)
    perte = nn.functional.binary_cross_entropy_with_logits(logits, annotations)
    exactitude = self.exactitudes[f"exactitude_{etape}"]
    exactitude(logits.sigmoid(), annotations.int())
    self.log(f"perte_{etape}", perte, on_step=False, on_epoch=True, prog_bar=True)
    self.log(
      f"exactitude_{etape}", exactitude, on_step=False, on_epoch=True, prog_bar=True
    )
    iou = self.ious[f"iou_{etape}"]
    iou(logits.sigmoid(), annotations.int())
    self.log(f"iou_{etape}", iou, on_step=False, on_epoch=True, prog_bar=True)
    return perte

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "entrainement")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], indice_lot: int
  ) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(
    self, batch: torch.Tensor | tuple[torch.Tensor, ...], indice_lot: int
  ) -> torch.Tensor:
    images = batch[0] if isinstance(batch, (list, tuple)) else batch
    return self(images).sigmoid().cpu()

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.hparams.taux_apprentissage)

In [ ]:
taille_lot = 4


def fabrique_chargeur(
  df: pandas.DataFrame, taille_lot: int, evaluation: bool
) -> DataLoader:
  """Un DataLoader sur les tuiles de `df` : batchs, mélange et préchargement."""
  return DataLoader(
    DatasetSegmentation(df, evaluation),
    batch_size=taille_lot,
    shuffle=not evaluation,
    num_workers=nb_lecteurs,
    persistent_workers=nb_lecteurs > 0,
  )


# Chargeurs d'entraînement et de validation : `nb_lecteurs` processus lisent les
# batchs suivants pendant que le GPU calcule
chargeur_entrainement = fabrique_chargeur(df_entrainement, taille_lot, evaluation=False)
chargeur_val = fabrique_chargeur(df_val, taille_lot, evaluation=True)

# Scores de test de chaque modèle, comparés à la fin du notebook
resultats_test: dict[str, dict[str, float]] = {}

Le notebook entraîne quatre modèles de la même façon. Écrivez la fonction `entraine(segmenteur, nom, nb_epoques)`, sur le modèle des slides « Entraîner avec le `Trainer` » et « Par early stopping ». Elle crée un `Trainer` de Lightning qui :

- entraîne `segmenteur` pendant `nb_epoques` epochs sur `chargeur_entrainement`, en l'évaluant sur `chargeur_val` à la fin de chaque epoch, sur le GPU s'il y en a un ;
- écrit les métriques de chaque epoch dans le dossier `journaux/<nom>` ;
- arrête l'entraînement quand la perte de validation n'a pas baissé depuis 5 epochs, et sauvegarde les poids de meilleure `iou_val`.

Elle renvoie ce `Trainer` : `trainer.test(..., ckpt_path="best")` rechargera ces meilleurs poids.

Entraînez ensuite le UNet de `construit_modele`, dans un `SegmenteurImages` de pas 1e-4, sous le nom `unet`. Lightning affiche la progression de chaque epoch.

**Documentation utile** :

- [`lightning.Trainer`](https://lightning.ai/docs/pytorch/stable/common/trainer.html)
- [`CSVLogger`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.loggers.csv_logs.html#lightning.pytorch.loggers.csv_logs.CSVLogger)
- [`EarlyStopping`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.callbacks.EarlyStopping.html)
- [`ModelCheckpoint`](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.callbacks.ModelCheckpoint.html)

In [ ]:
def entraine(
  segmenteur: SegmenteurImages, nom: str, nb_epoques: int = nb_epoques
) -> lightning.Trainer:
  # Votre code ici
  ...


# Votre code ici : le UNet de construit_modele, entraîné avec un pas de 1e-4

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `lightning.Trainer(max_epochs=nb_epoques, accelerator="auto", devices=1, logger=..., callbacks=[...])` : `accelerator="auto"` choisit le GPU s'il y en a un
- `CSVLogger("journaux", name=nom)` : les métriques de chaque epoch, dans `journaux/<nom>/version_0/metrics.csv`
- `TQDMProgressBar()`, en tête des callbacks : la barre de progression de tqdm, plutôt que celle de rich, que Lightning choisit quand rich est installé (Colab) et qui a figé des entraînements
- `EarlyStopping(monitor="perte_val", patience=5)`
- `ModelCheckpoint(monitor="iou_val", mode="max", save_weights_only=True, save_top_k=1)` : `mode="max"`, puisqu'une IoU plus grande est meilleure ; `save_top_k=1` ne garde que le meilleur fichier
- `trainer.fit(segmenteur, chargeur_entrainement, chargeur_val)`
- `ModelSummary(segmenteur, max_depth=2)` affiche les couches du modèle, la forme de leurs sorties et le nombre de paramètres

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def entraine(
  segmenteur: SegmenteurImages, nom: str, nb_epoques: int = nb_epoques
) -> lightning.Trainer:
  trainer = lightning.Trainer(
    max_epochs=nb_epoques,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("journaux", name=nom),
    callbacks=[
      TQDMProgressBar(),
      EarlyStopping(monitor=..., patience=5),
      ModelCheckpoint(monitor=..., mode=..., save_weights_only=True, save_top_k=1),
    ],
  )
  trainer.fit(..., chargeur_entrainement, chargeur_val)
  return trainer


segmenteur = SegmenteurImages(construit_modele(), taux_apprentissage=...)
print(ModelSummary(segmenteur, max_depth=2))
trainer = entraine(segmenteur, "unet")
```

### Solution

In [ ]:
def entraine(
  segmenteur: SegmenteurImages, nom: str, nb_epoques: int = nb_epoques
) -> lightning.Trainer:
  """Entraîne `segmenteur` et garde les poids de meilleure IoU de validation."""
  trainer = lightning.Trainer(
    max_epochs=nb_epoques,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("journaux", name=nom),
    callbacks=[
      # La barre de progression de tqdm, plutôt que celle de rich que Lightning
      # choisit quand rich est installé (Colab) : elle a figé des entraînements
      TQDMProgressBar(),
      EarlyStopping(monitor="perte_val", patience=5),
      # Les poids de meilleure IoU de validation, que `trainer.test` rechargera
      ModelCheckpoint(
        monitor="iou_val", mode="max", save_weights_only=True, save_top_k=1
      ),
    ],
  )
  # Évaluation sur le chargeur de validation à la fin de chaque epoch
  trainer.fit(segmenteur, chargeur_entrainement, chargeur_val)
  return trainer


segmenteur = SegmenteurImages(construit_modele(), taux_apprentissage=1e-4)
print(ModelSummary(segmenteur, max_depth=2))
trainer = entraine(segmenteur, "unet")

## Application aux données de test

La cellule suivante crée le chargeur de test, et place dans `X` et `Y` 8 tuiles de test qui ont des lisières : entre 20 et 80 % de forêt dans la partie recadrée que voit le modèle.

Évaluez le modèle sur tout le chargeur de test, avec les poids de sa meilleure epoch, et rangez ses scores dans `resultats_test["UNet entraîné depuis zéro"]`. Affichez ensuite ses prédictions pour les 8 tuiles de `X` avec `affiche_echantillon`.

Les prédictions sont-elles de bonne qualité ? Comparez l'IoU de test à celle d'un modèle qui ne prédirait jamais la forêt, et regardez où se trouvent les erreurs.

**Documentation utile** :

- [`Trainer.test`](https://lightning.ai/docs/pytorch/stable/common/trainer.html#test)
- [`Trainer.predict`](https://lightning.ai/docs/pytorch/stable/common/trainer.html#predict)

In [ ]:
chargeur_test = fabrique_chargeur(df_test, 8, evaluation=True)

# 8 tuiles de test qui ont des lisières : entre 20 et 80 % de forêt dans la partie
# recadrée que voit le modèle
recadrage = (dim_originale - dim_entree) // 2


def calcule_part_foret(chemin: str) -> float:
  foret = lit_annotation(chemin)[
    0, recadrage : recadrage + dim_entree, recadrage : recadrage + dim_entree
  ]
  return foret.mean().item()


parts = df_test.Annotations.map(calcule_part_foret)
df_lisieres = df_test[parts.between(0.2, 0.8)].sample(8, random_state=0)
chargeur_lisieres = DataLoader(
  DatasetSegmentation(df_lisieres, evaluation=True), batch_size=8
)
X, Y = next(iter(chargeur_lisieres))
X.shape, Y.shape

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `trainer.test(segmenteur, dataloaders=chargeur_test, ckpt_path="best")` : recharge dans `segmenteur` les poids sauvegardés par le `ModelCheckpoint`, puis renvoie une liste d'un dictionnaire par chargeur, `[{"perte_test": …, "exactitude_test": …, "iou_test": …}]`
- `trainer.predict(segmenteur, chargeur_lisieres)` : une liste de tenseurs de probabilités de forêt, un par batch ; `torch.cat(...)` les met bout à bout, en un tenseur de forme (8, 1, 512, 512)
- `for x, y, p in zip(X, Y, predictions):` parcourt les images, les cibles et les prédictions ensemble

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
scores = trainer.test(segmenteur, dataloaders=..., ckpt_path="best")[0]
resultats_test["UNet entraîné depuis zéro"] = scores
print(resultats_test)
predictions = torch.cat(trainer.predict(segmenteur, ...))
for x, y, p in zip(X, Y, predictions):
  affiche_echantillon(x, y, p)
```

### Solution

In [ ]:
# `ckpt_path="best"` recharge les poids de meilleure IoU de validation
scores = trainer.test(segmenteur, dataloaders=chargeur_test, ckpt_path="best")[0]
resultats_test["UNet entraîné depuis zéro"] = scores
print(resultats_test)

# Probabilités de forêt de chaque pixel des 8 tuiles : `predict` renvoie un tenseur
# par batch, mis bout à bout
predictions = torch.cat(trainer.predict(segmenteur, chargeur_lisieres))
for x, y, p in zip(X, Y, predictions):
  affiche_echantillon(x, y, p)

Un modèle qui ne prédit jamais la forêt a une IoU nulle, quelle que soit son exactitude : l'IoU de test du UNet en est loin, et les grandes zones de forêt et de non-forêt sont bien détourées. Les erreurs se concentrent :

- le long des lisières, où la prédiction hésite (probabilités intermédiaires) et où la cible elle-même n'est précise qu'à quelques pixels près ;
- sur les petits objets, haies, bosquets, clairières ou chemins forestiers, que la définition de l'IGN (plus de 5 000 m², plus de 20 m de large) ne classe pas toujours comme l'œil ;
- sur la végétation qui n'est pas de la forêt, prairies et cultures, rouge elle aussi en infrarouge couleur.

## Modèle pré-entraîné sans connexions de saut

Le UNet précédent apprend tout depuis zéro, encodeur compris, sur 1 296 tuiles. Un encodeur pré-entraîné sur ImageNet sait déjà extraire des bords, des textures et des formes : c'est l'apprentissage par transfert (slides « Transfer learning »). ImageNet ne contient pas d'images infrarouges, mais ces caractéristiques visuelles restent utiles.

La cellule suivante charge EfficientNetV2-B0 avec `timm`, sans sa tête de classification, et fournit deux modules :

- `Normalise` normalise les pixels, déjà ramenés entre 0 et 1, avec la moyenne et l'écart-type de chaque canal sur ImageNet : les entrées que le réseau a vues pendant son entraînement ;
- `Gele` gèle un module, comme dans la slide « Geler aussi les BatchNorm : `Gele` » : ses poids ne bougent plus, et ses BatchNorm gardent les statistiques d'ImageNet. Lightning garde le mode, `train()` ou `eval()`, de chaque sous-module : le module gelé reste en `eval()` pendant tout l'entraînement. La méthode `train` de `Gele` sert dans une boucle écrite à la main, qui appelle `modele.train()` à chaque epoch.

Pour une tuile de 512 × 512, EfficientNetV2-B0 rend des cartes de 16 × 16 × 1280.

Écrivez un décodeur volontairement simple, **sans connexions de saut** : il ne fait que suréchantillonner la sortie de l'encodeur gelé, de 16 × 16 jusqu'à 512 × 512. Ce modèle montre ce qu'on perd sans elles : l'encodeur reconnaît la forêt, mais les détails fins calculés par ses premières couches n'atteignent jamais le décodeur.

Entraînez-le avec `entraine`, avec un pas de 1e-3 (seul le décodeur, parti de zéro, apprend) pendant `nb_epoques_pretraine` epochs, sous le nom `sans_saut`. Rangez ses scores de test dans `resultats_test["EfficientNetV2-B0 sans connexions de saut"]`, et comparez ses prédictions sur `X` à celles du UNet.

**Documentation utile** :

- [`nn.Upsample`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html)

In [ ]:
class Normalise(nn.Module):
  """Normalise, canal par canal, des images aux pixels entre 0 et 1."""

  def __init__(
    self, moyenne: tuple[float, ...] | float, ecart_type: tuple[float, ...] | float
  ) -> None:
    super().__init__()
    self.register_buffer("moyenne", torch.tensor(moyenne).reshape(1, -1, 1, 1))
    self.register_buffer("ecart_type", torch.tensor(ecart_type).reshape(1, -1, 1, 1))

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return (images - self.moyenne) / self.ecart_type


class Gele(nn.Module):
  """Un module aux poids gelés, gardé en mode évaluation."""

  def __init__(self, module: nn.Module) -> None:
    super().__init__()
    self.module = module.requires_grad_(False)
    self.eval()

  def train(self, mode: bool = True) -> "Gele":
    # BatchNorm et dropout restent en mode évaluation, même quand une boucle
    # écrite à la main appelle modele.train()
    return super().train(False)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.module(images)


# EfficientNetV2-B0 pré-entraîné sur ImageNet, sans sa tête de classification
modele_base = timm.create_model(
  "tf_efficientnetv2_b0.in1k", pretrained=True, num_classes=0, global_pool=""
)
# Moyenne et écart-type de chaque canal sur ImageNet
config_donnees = timm.data.resolve_model_data_config(modele_base)
config_donnees["mean"], config_donnees["std"], modele_base.num_features

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Sequential(Normalise(config_donnees["mean"], config_donnees["std"]), Gele(modele_base), ...)` : la normalisation, l'encodeur gelé, puis les couches du décodeur
- Une convolution 1 × 1 suivie d'une ReLU réduit d'abord les 1280 canaux à 512
- Cinq fois de suite : `nn.Upsample(scale_factor=2)`, qui double la résolution, puis une convolution 3 × 3 (`padding="same"`) suivie d'une ReLU, avec 256, 128, 64, 32 puis 16 canaux en sortie
- `nn.Dropout(0.1)`, puis `nn.Conv2d(16, 1, 3, padding="same")` : le logit de chaque pixel
- Puis les mêmes appels que pour le UNet : `SegmenteurImages(modele, taux_apprentissage=1e-3)`, `entraine`, `trainer.test`, `trainer.predict`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def cree_convolution(
  canaux_entree: int, canaux_sortie: int, taille_noyau: int = 3
) -> nn.Sequential:
  return nn.Sequential(
    nn.Conv2d(canaux_entree, canaux_sortie, taille_noyau, padding="same"), nn.ReLU()
  )


decodeur = [
  cree_convolution(modele_base.num_features, 512, taille_noyau=1)
]  # 16 × 16 × 512
for canaux_entree, canaux_sortie in zip(
  [512, 256, 128, 64, 32], [256, 128, 64, 32, 16]
):
  decodeur += [
    ...,
    cree_convolution(canaux_entree, canaux_sortie),
  ]  # résolution doublée
modele = nn.Sequential(
  Normalise(config_donnees["mean"], config_donnees["std"]),
  Gele(modele_base),  # 16 × 16 × 1280
  *decodeur,  # 512 × 512 × 16
  nn.Dropout(0.1),
  ...,  # un logit par pixel
)

segmenteur = SegmenteurImages(modele, taux_apprentissage=1e-3)
print(ModelSummary(segmenteur, max_depth=2))
trainer = entraine(segmenteur, "sans_saut", nb_epoques=nb_epoques_pretraine)
scores = ...
resultats_test["EfficientNetV2-B0 sans connexions de saut"] = scores
predictions = torch.cat(trainer.predict(segmenteur, chargeur_lisieres))
for x, y, p in zip(X, Y, predictions):
  affiche_echantillon(x, y, p)
```

### Solution

In [ ]:
def cree_convolution(
  canaux_entree: int, canaux_sortie: int, taille_noyau: int = 3
) -> nn.Sequential:
  """Une convolution suivie d'une ReLU, qui garde la taille de l'image."""
  return nn.Sequential(
    nn.Conv2d(canaux_entree, canaux_sortie, taille_noyau, padding="same"), nn.ReLU()
  )


# Décodeur : de 16 × 16 à 512 × 512, en doublant 5 fois la résolution, sans rien
# recevoir des couches intermédiaires de l'encodeur
decodeur = [
  cree_convolution(modele_base.num_features, 512, taille_noyau=1)
]  # 16 × 16 × 512
for canaux_entree, canaux_sortie in zip(
  [512, 256, 128, 64, 32], [256, 128, 64, 32, 16]
):
  decodeur += [
    nn.Upsample(scale_factor=2),
    cree_convolution(canaux_entree, canaux_sortie),
  ]
modele = nn.Sequential(
  Normalise(config_donnees["mean"], config_donnees["std"]),
  Gele(modele_base),  # 16 × 16 × 1280
  *decodeur,  # 512 × 512 × 16
  nn.Dropout(0.1),
  nn.Conv2d(16, 1, 3, padding="same"),
)

# Seul le décodeur, parti de zéro, apprend : un pas plus grand que pour le UNet
segmenteur = SegmenteurImages(modele, taux_apprentissage=1e-3)
print(ModelSummary(segmenteur, max_depth=2))
trainer = entraine(segmenteur, "sans_saut", nb_epoques=nb_epoques_pretraine)
scores = trainer.test(segmenteur, dataloaders=chargeur_test, ckpt_path="best")[0]
resultats_test["EfficientNetV2-B0 sans connexions de saut"] = scores
print(resultats_test)

# Les mêmes tuiles de test que pour le UNet
predictions = torch.cat(trainer.predict(segmenteur, chargeur_lisieres))
for x, y, p in zip(X, Y, predictions):
  affiche_echantillon(x, y, p)

Comparez ses masques à ceux du UNet, et le détail des contours à celui de la cible : le décodeur reconstruit les contours à partir de cartes de 16 × 16 seulement, où un pixel couvre 32 × 32 pixels de la tuile. L'encodeur pré-entraîné reconnaît la forêt, mais sans connexions de saut, le décodeur sait moins précisément *où* passent ses bords : cela se voit sur les lisières, pas forcément dans l'IoU.

## UNet pré-entraîné

Combinons les deux : un vrai UNet, avec connexions de saut, dont l'encodeur est le même EfficientNetV2-B0 pré-entraîné (slide « UNet avec un encodeur pré-entraîné »). La bibliothèque `segmentation_models_pytorch` (smp) l'assemble : seules les connexions de saut le distinguent du modèle précédent.

1. Créez le UNet de smp, avec l'encodeur `tu-tf_efficientnetv2_b0` pré-entraîné sur ImageNet (le préfixe `tu-` désigne un modèle de `timm`) et une seule classe en sortie.
2. Gelez son encodeur avec `Gele` : seul le décodeur apprend d'abord.
3. Affichez les formes des sorties de l'encodeur pour une tuile de 512 × 512 : ce sont elles que smp relie au décodeur par les connexions de saut. À quelles résolutions sont-elles prises ?
4. Placez `Normalise` devant, comme pour le modèle précédent.
5. Entraînez-le comme le précédent, sous le nom `unet_pretraine`, puis rangez ses scores de test dans `resultats_test["UNet pré-entraîné, encodeur gelé"]`, et comparez ses prédictions sur `X` à celles des deux modèles précédents.

**Documentation utile** :

- [`smp.Unet`](https://smp.readthedocs.io/en/latest/models.html#unet)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `smp.Unet(encoder_name="tu-tf_efficientnetv2_b0", encoder_weights=True, classes=1)` : un logit par pixel en sortie ; pour un encodeur `tu-`, `encoder_weights=True` charge les poids pré-entraînés que désigne le nom du modèle
- `unet.encoder = Gele(unet.encoder)` : gèle l'encodeur, ses poids comme les statistiques de ses BatchNorm
- `unet.encoder(torch.zeros(1, nb_canaux, dim_entree, dim_entree))`, dans un bloc `with torch.no_grad():` : la liste des sorties de l'encodeur, l'image elle-même en tête ; appelé une fois l'encodeur gelé, en mode évaluation, il ne touche pas aux statistiques de ses BatchNorm
- `nn.Sequential(Normalise(config_donnees["mean"], config_donnees["std"]), unet)` : les pixels normalisés comme ceux d'ImageNet, puis le UNet
- `SegmenteurImages(unet_pretraine, taux_apprentissage=1e-3)`, puis `entraine(segmenteur, "unet_pretraine", nb_epoques=nb_epoques_pretraine)` : seul le décodeur, parti de zéro, apprend
- `trainer.test(..., ckpt_path="best")` recharge aussi les meilleurs poids dans `unet_pretraine` : le fine-tuning repartira de là

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
unet = smp.Unet(encoder_name=..., encoder_weights=True, classes=1)
unet.encoder = ...  # encodeur gelé
with torch.no_grad():
  caracteristiques = unet.encoder(torch.zeros(1, nb_canaux, dim_entree, dim_entree))
print([tuple(caracteristique.shape) for caracteristique in caracteristiques])
unet_pretraine = nn.Sequential(
  Normalise(config_donnees["mean"], config_donnees["std"]), unet
)

segmenteur = SegmenteurImages(unet_pretraine, taux_apprentissage=1e-3)
print(ModelSummary(segmenteur, max_depth=2))
trainer = entraine(segmenteur, "unet_pretraine", nb_epoques=nb_epoques_pretraine)
scores = ...
resultats_test["UNet pré-entraîné, encodeur gelé"] = scores
predictions = ...
for x, y, p in zip(X, Y, predictions):
  affiche_echantillon(x, y, p)
```

### Solution

In [ ]:
# Pour un encodeur `tu-` (timm), encoder_weights=True : les poids pré-entraînés
# que désigne le nom du modèle
unet = smp.Unet(encoder_name="tu-tf_efficientnetv2_b0", encoder_weights=True, classes=1)
# Étape 1 : l'encodeur est gelé (poids et statistiques des BatchNorm), seul le
# décodeur apprend
unet.encoder = Gele(unet.encoder)
# Sorties de l'encodeur reliées au décodeur : l'image, la dernière sortie à 256,
# 128, 64 et 32 pixels de côté, puis le fond du « U », à 16 × 16 (192 canaux :
# smp s'arrête au dernier bloc, avant la convolution finale à 1 280 canaux)
with torch.no_grad():
  caracteristiques = unet.encoder(torch.zeros(1, nb_canaux, dim_entree, dim_entree))
print([tuple(caracteristique.shape) for caracteristique in caracteristiques])
unet_pretraine = nn.Sequential(
  Normalise(config_donnees["mean"], config_donnees["std"]), unet
)

segmenteur = SegmenteurImages(unet_pretraine, taux_apprentissage=1e-3)
print(ModelSummary(segmenteur, max_depth=2))
trainer = entraine(segmenteur, "unet_pretraine", nb_epoques=nb_epoques_pretraine)
# `ckpt_path="best"` recharge aussi les meilleurs poids dans `unet_pretraine`, d'où
# repartira le fine-tuning
scores = trainer.test(segmenteur, dataloaders=chargeur_test, ckpt_path="best")[0]
resultats_test["UNet pré-entraîné, encodeur gelé"] = scores
print(resultats_test)

predictions = torch.cat(trainer.predict(segmenteur, chargeur_lisieres))
for x, y, p in zip(X, Y, predictions):
  affiche_echantillon(x, y, p)

### Fine-tuning de l'encodeur

Une fois le décodeur entraîné, dégelez l'encodeur, et poursuivez l'apprentissage de tout le réseau pendant `nb_epoques_affinage` epochs (slides « Fine-tuning : dégeler le backbone » et « Fine-tuning : garder les BatchNorm gelées »), avec AdamW et deux groupes de paramètres :

- un pas de 1e-5 pour l'encodeur, dont les poids sont déjà bons : un grand pas les détruirait ;
- un pas de 1e-4 pour le décodeur, déjà entraîné : 10 fois plus petit qu'à l'étape précédente.

Pour cela, écrivez `SegmenteurAffine`, un `SegmenteurImages` dont `configure_optimizers` crée ces deux groupes. Retirez ensuite `Gele`, et remettez le réseau en mode entraînement, sauf les BatchNorm de l'encodeur, qui restent gelées : sinon, elles recalculent leurs statistiques sur nos batchs de 4 tuiles infrarouges, bien différentes de celles d'ImageNet, et dégradent l'encodeur. Lightning garde le mode, `train()` ou `eval()`, que vous donnez à chaque sous-module : des BatchNorm mises en `eval()` y restent pendant tout l'entraînement.

Entraînez le modèle sous le nom `unet_affine`, puis rangez ses scores de test dans `resultats_test["UNet pré-entraîné, fine-tuné"]`.

**Documentation utile** :

- [Options par groupe de paramètres](https://docs.pytorch.org/docs/stable/optim.html#per-parameter-options)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `self.named_parameters()` : les paramètres du modèle, avec leur nom ; ceux de l'encodeur ont `.encoder.` dans leur nom
- `torch.optim.AdamW([{"params": encodeur, "lr": 1e-5}, {"params": autres, "lr": 1e-4}])` : un pas par groupe de paramètres
- `self.hparams.taux_apprentissage` : le pas passé au constructeur d'`SegmenteurImages`
- `unet.encoder = unet.encoder.module.requires_grad_(True)` : retire `Gele`, et rend les poids de l'encodeur entraînables
- `unet_pretraine.train()` : tout le réseau en mode entraînement, encodeur compris
- `for module in unet.encoder.modules():`, puis `if isinstance(module, nn.BatchNorm2d):` et `module.eval().requires_grad_(False)` : regèle les BatchNorm de l'encodeur, poids et statistiques ; celles de timm (`BatchNormAct2d`) héritent de `nn.BatchNorm2d`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class SegmenteurAffine(SegmenteurImages):
  def __init__(
    self, modele: nn.Module, taux_apprentissage_encodeur: float = 1e-5, **kwargs
  ) -> None:
    super().__init__(modele, **kwargs)
    self.taux_apprentissage_encodeur = taux_apprentissage_encodeur

  def configure_optimizers(self) -> torch.optim.Optimizer:
    parameters = list(self.named_parameters())
    encodeur = [p for nom, p in parameters if ".encoder." in nom]
    autres = ...
    return torch.optim.AdamW(
      [
        {"params": encodeur, "lr": ...},
        {"params": autres, "lr": self.hparams.taux_apprentissage},
      ]
    )


unet.encoder = ...  # encodeur sans Gele, poids entraînables
unet_pretraine.train()
for module in unet.encoder.modules():
  if isinstance(module, ...):
    module.eval().requires_grad_(...)  # BatchNorm regelées
segmenteur = SegmenteurAffine(
  unet_pretraine, taux_apprentissage_encodeur=1e-5, taux_apprentissage=1e-4
)
trainer = entraine(segmenteur, "unet_affine", nb_epoques=nb_epoques_affinage)
scores = ...
resultats_test["UNet pré-entraîné, fine-tuné"] = scores
```

#### Solution

In [ ]:
class SegmenteurAffine(SegmenteurImages):
  """Un SegmenteurImages dont l'encodeur a son propre pas, plus petit."""

  def __init__(
    self, modele: nn.Module, taux_apprentissage_encodeur: float = 1e-5, **kwargs
  ) -> None:
    super().__init__(modele, **kwargs)
    self.taux_apprentissage_encodeur = taux_apprentissage_encodeur

  def configure_optimizers(self) -> torch.optim.Optimizer:
    parameters = list(self.named_parameters())
    encodeur = [p for nom, p in parameters if ".encoder." in nom]
    autres = [p for nom, p in parameters if ".encoder." not in nom]
    return torch.optim.AdamW(
      [
        {"params": encodeur, "lr": self.taux_apprentissage_encodeur},
        {"params": autres, "lr": self.hparams.taux_apprentissage},
      ]
    )


# Étape 2 : l'encodeur sans Gele, ses poids de nouveau entraînables
unet.encoder = unet.encoder.module.requires_grad_(True)
# Tout le réseau en mode entraînement (dropout actif)...
unet_pretraine.train()
# ...sauf les BatchNorm de l'encodeur, regelées : peu de données et des batchs de
# 4 tuiles, les statistiques d'ImageNet sont plus fiables que celles des batchs.
# Lightning garde le mode de chaque sous-module : elles restent en eval()
for module in unet.encoder.modules():
  if isinstance(module, nn.BatchNorm2d):
    module.eval().requires_grad_(False)
# Un pas 10 fois plus petit pour l'encodeur : ajuster ses poids sans les détruire
segmenteur = SegmenteurAffine(
  unet_pretraine, taux_apprentissage_encodeur=1e-5, taux_apprentissage=1e-4
)
trainer = entraine(segmenteur, "unet_affine", nb_epoques=nb_epoques_affinage)
scores = trainer.test(segmenteur, dataloaders=chargeur_test, ckpt_path="best")[0]
resultats_test["UNet pré-entraîné, fine-tuné"] = scores
print(resultats_test)

## Comparaison des modèles

La cellule suivante compare les scores de test des quatre modèles, ainsi que l'exactitude d'un modèle qui répondrait toujours la classe la plus fréquente des tuiles de test. Quel modèle obtient la meilleure IoU ? Les écarts entre les autres sont-ils assez grands pour les classer ? Pourquoi les écarts d'exactitude entre les modèles sont-ils plus faibles que les écarts d'IoU ?

In [ ]:
# Part des pixels de forêt dans la partie recadrée des tuiles de test
foret = parts.mean()
print(
  "Exactitude en répondant toujours la classe la plus fréquente :",
  f"{max(foret, 1 - foret):.3f}",
)
pandas.DataFrame(resultats_test).T[["iou_test", "exactitude_test", "perte_test"]].round(
  3
)

### Solution

Le UNet pré-entraîné et fine-tuné arrive en tête, ou à égalité en tête, à chaque exécution : il combine un encodeur qui a appris à reconnaître des textures sur ImageNet, les connexions de saut, qui gardent les contours précis, et un réglage de tout le réseau sur nos images infrarouges. Les trois autres modèles sont à moins d'un point d'IoU les uns des autres (entre 0,965 et 0,968 dans nos deux exécutions), et leur ordre change d'une exécution à l'autre : sur cette tâche facile (forêt ou non-forêt), ces écarts sont de l'ordre de la variation d'un entraînement à l'autre. Ils ne permettent donc pas de dire si les connexions de saut, le pré-entraînement ou la quantité de données font la différence. Pour les départager, il faudrait répéter chaque entraînement avec plusieurs graines et comparer les moyennes, ou regarder les tuiles à lisières affichées plus haut, où les erreurs se voient.

L'exactitude compte tous les pixels, dont la grande majorité est facile : l'intérieur des zones de forêt ou de non-forêt. Elle se lit mal sans l'exactitude du modèle qui répond toujours la classe la plus fréquente : tous les modèles la dépassent largement, ce qui montre seulement qu'ils ont appris quelque chose, et leurs valeurs, serrées près de 1, cachent comment ils se trompent. L'IoU ne compte que les pixels que la prédiction ou la cible disent de la forêt : chaque pixel de forêt manqué ou en trop y pèse davantage, et elle étale donc un peu plus les écarts que l'exactitude. Elle les rend plus lisibles, sans les rendre significatifs pour autant.